# PBMC 3k scRNA-seq Walkthrough

End-to-end single-cell RNA-seq analysis using the public 10x Genomics PBMC 3k dataset.

**Prerequisites**
- Run `bash download_pbmc3k.sh` to obtain raw data
- Run CellRanger count (see README.md) or point `MATRIX_DIR` at an existing output
- Install dependencies: `pip install scanpy anndata matplotlib pandas`

**Pipeline steps**
1. Load CellRanger output
2. QC filtering (low-quality cells, high mitochondrial content)
3. Normalisation and highly variable gene selection
4. PCA → kNN graph → UMAP
5. Leiden clustering
6. Marker gene identification

In [ ]:
from pathlib import Path

# ── Configuration — adjust these paths ──────────────────────────────────────
MATRIX_DIR = Path("data/pbmc3k/outs/filtered_feature_bc_matrix")
OUTPUT_DIR = Path("output/pbmc3k")

# QC / pipeline parameters
MIN_GENES      = 200
MIN_CELLS      = 3
MAX_PCT_MT     = 20.0
N_TOP_GENES    = 2000
N_NEIGHBORS    = 15

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Matrix dir : {MATRIX_DIR}")
print(f"Output dir : {OUTPUT_DIR}")

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.figdir = str(OUTPUT_DIR)
print(f"scanpy {sc.__version__}")

## 1. Load CellRanger output

In [ ]:
adata = sc.read_10x_mtx(str(MATRIX_DIR), var_names="gene_symbols", cache=False)
adata.var_names_make_unique()
print(adata)

## 2. QC filtering

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4, multi_panel=True,
)

# Apply filters
adata = adata[adata.obs.n_genes_by_counts >= MIN_GENES, :]
adata = adata[adata.obs.pct_counts_mt < MAX_PCT_MT, :]
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
print(f"After QC: {adata.n_obs} cells, {adata.n_vars} genes")

## 3. Normalisation and HVG selection

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES)
sc.pl.highly_variable_genes(adata)
adata = adata[:, adata.var.highly_variable]
print(f"Selected {adata.n_vars} highly variable genes")

## 4. PCA → kNN → UMAP

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, log=True)

sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=40)
sc.tl.umap(adata)

## 5. Leiden clustering

In [ ]:
sc.tl.leiden(adata)
sc.pl.umap(adata, color="leiden", legend_loc="on data", title="PBMC 3k — Leiden clusters")
clusters = sorted(adata.obs["leiden"].unique(), key=int)
print(f"Found {len(clusters)} clusters: {clusters}")

## 6. Marker genes

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

marker_df = sc.get.rank_genes_groups_df(adata, group=None)
marker_df.to_csv(OUTPUT_DIR / "marker_genes.csv", index=False)
marker_df.head(20)

## 7. Save results

In [ ]:
import json

adata.write(OUTPUT_DIR / "cells.h5ad")

cells_per_cluster = {c: int((adata.obs["leiden"] == c).sum()) for c in clusters}
cluster_summary = {"n_clusters": len(clusters), "cells_per_cluster": cells_per_cluster}
with open(OUTPUT_DIR / "cluster_summary.json", "w") as fh:
    json.dump(cluster_summary, fh, indent=2)

print("Outputs written to:", OUTPUT_DIR)
print("  cells.h5ad")
print("  marker_genes.csv")
print("  cluster_summary.json")
print(json.dumps(cluster_summary, indent=2))